# Reload

In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
import random
import datetime

from time_slot import TimeSlot
from locations import Locations
from truck import *
from freight import Freight

#Imports for the AI algorithms
import numpy as np
import gymnasium as gym
from gymnasium import spaces

import os
import time

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.ppo import MlpPolicy

#Imports for the imitation learning
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.util.util import make_vec_env

from stable_baselines3.common.evaluation import evaluate_policy

from imitation.data import rollout

from imitation.algorithms import bc

from imitation.algorithms import preference_comparisons
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from imitation.policies.base import FeedForward32Policy, NormalizeFeaturesExtractor
from imitation.rewards.reward_wrapper import RewardVecEnvWrapper

from imitation.data.types import Trajectory, Transitions
import pickle

from imitation.algorithms.adversarial.gail import GAIL
from imitation.algorithms.adversarial.airl import AIRL
from imitation.rewards.reward_nets import BasicShapedRewardNet

import tempfile
from imitation.algorithms.dagger import SimpleDAggerTrainer

#Explainability
import sklearn
import sklearn.datasets
import sklearn.ensemble
import lime
import lime.lime_tabular
from __future__ import print_function
import pandas as pd
from sklearn.utils import Bunch
import torch

#S2E
from S2E_rewards import Rewards

# Parameters

In [3]:
number_of_trucks = 125
number_of_freight = 250

# Time Frame

In [4]:
start_time = datetime.datetime.now() 
print("start_time =", start_time)

#Time frame of 1 week
number_of_days = 7

delta_time = datetime.timedelta(days=1) 
delta_time_seconds_day = 24 * 60 * 60 #Seconds in a day
delta_time_seconds_week = number_of_days * 24 * 60 * 60 #Seconds in a Week

time_frame = []

for i in range (number_of_days):
    end_time = start_time + delta_time
    
    time_frame += [start_time]
    start_time = end_time

#Return start_time back to normal
start_time -= datetime.timedelta(seconds=delta_time_seconds_week)
print("start_time =", start_time)
    
print("time_frame =", time_frame, end = '\n\n')

days_of_the_week = []
for i in range(number_of_days):
    days_of_the_week.append(i)

start_time = 2026-01-26 12:01:14.625569
start_time = 2026-01-26 12:01:14.625569
time_frame = [datetime.datetime(2026, 1, 26, 12, 1, 14, 625569), datetime.datetime(2026, 1, 27, 12, 1, 14, 625569), datetime.datetime(2026, 1, 28, 12, 1, 14, 625569), datetime.datetime(2026, 1, 29, 12, 1, 14, 625569), datetime.datetime(2026, 1, 30, 12, 1, 14, 625569), datetime.datetime(2026, 1, 31, 12, 1, 14, 625569), datetime.datetime(2026, 2, 1, 12, 1, 14, 625569)]



# Locations

In [5]:
locations = Locations.locations

# Trucks

In [6]:
def generateRandomTrucks(number_of_trucks):
    trucks = []
    
    #Full list of haulier companies at at https://www.imt-ip.pt/sites/IMTT/Portugues/TransportesRodoviarios/EmpresasLicenciadas/Documents/Empresas_Mercadorias_OrdEmp.pdf
    companies = ["CARKARMA", "CASTRO E MERINO TRANSPORTES", "DESTINOS VIBRANTES, UNIPESSOAL", "DROPLOG", "ESTORILMUDA - TRANSPORTES", "MAIN ROAD CONSTRUCTIONS", "SILVA & AZEVEDO", "TRANSPORTES JOAQUIM MONTEIRO", "TRANSPORTES MALHADA", "TRANSPORTES RAIO", "WINDY TRANSPORT-SUCURSAL EM PORTUGAL"]
    
    truck_types = ["20' Standard - Steel", "40' Standard - Steel", "40' High - Steel", "45' High - Steel", "20' Standard Reefer", "40' High Cube Standard Reefer", "40' High Cube Magnum Reefer", "40' High Cube StarCare", "40' High Cube Super Freezer", "20' Open Top", "40' Open Top High Cube", "40' High Flat Rack Collapsible"]

    for current_id in range(number_of_trucks):
        company = random.choice(companies)
        location = random.choice(locations)

        availability = {}
        
        number_of_time_slots = random.randint(3, 7) #We are considering the time frame of 1 week
        time_ids = random.sample(days_of_the_week, k = number_of_time_slots)

        for time_id in time_ids:
            #from https://stackoverflow.com/questions/553303/how-to-generate-a-random-date-between-two-other-dates?page=1&tab=scoredesc#tab-top
            random_second_start = random.randrange(delta_time_seconds_day) #Random time between the two dates
            new_time_slot_start = time_frame[time_id] + datetime.timedelta(seconds=random_second_start)

            random_second_end = random.randrange(delta_time_seconds_day) #Random time between the two dates
            new_time_slot_end = time_frame[time_id] + datetime.timedelta(seconds=random_second_end)

            if random_second_start < random_second_end: #Start has to come before the end
                new_time_slot = TimeSlot(time_id, new_time_slot_start, new_time_slot_end)
            else: #Start doesn't come before the end, so switch places
                new_time_slot = TimeSlot(time_id, new_time_slot_end, new_time_slot_start)

            availability[time_id] = new_time_slot

        #emission rate in gCO2/t-km; between 58.3 and 307.2 according to https://theicct.org/publication/co2-emissions-from-trucks-in-the-eu-an-analysis-of-the-heavy-duty-co2-standards-baseline-data/
        emission_rate = round(random.uniform(58.3, 307.3), 1)
        
        #Choose type of truck
        #truck_type = random.choice(truck_types)
        truck_type = random.choices(truck_types, cum_weights = [1, 2, 3, 4, 4.8, 5.6, 6.4, 7.2, 8, 9.33, 10.66, 11.99], k = 1)[0]
        
        if (truck_type == "20' Standard - Steel"):
            truck = Dry_20StandardSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' Standard - Steel"):
            truck = Dry_40StandardSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High - Steel"):
            truck = Dry_40HighSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "45' High - Steel"):
            truck = Dry_45HighSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "20' Standard Reefer"):
            truck = Refrigerated_20StandardReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Standard Reefer"):
            truck = Refrigerated_40HighCubeStandardReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Magnum Reefer"):
            truck = Refrigerated_40HighCubeMagnumReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube StarCare"):
            truck = Refrigerated_40HighCubeStarCare(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Super Freezer"):
            truck = Refrigerated_40HighCubeSuperFreezer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "20' Open Top"):
            truck = Special_20OpenTop(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' Open Top High Cube"):
            truck = Special_40OpenTopHighCube(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Flat Rack Collapsible"):
            truck = Special_40HighFlatRackCollapsible(current_id, company, location, availability, emission_rate)
        
        trucks.append(truck)
        
    return trucks
    


# Freight

In [7]:
def generateRandomFreight(number_of_freight):
    freight_list = []
    
    categories = ["Dry", "Refrigerated", "Special"]

    current_id = 0

    for i in range(number_of_freight):
        category = random.choice(categories)
        
        if (category == "Dry"):
            weight = random.uniform(24, 28.87) #28.87 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 13555) #13556 = length of the container with the most length
            width = random.uniform(2000, 2351) #2352 = width of the widest container
            height = random.uniform(1800, 2697) #2698 = height of the tallest container
        elif (category == "Refrigerated"):
            weight = random.uniform(24, 29.67) #29.67 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 11587) #11588 = length of the container with the most length
            width = random.uniform(2200, 2289) #2290 = width of the widest container
            height = random.uniform(1700, 2504) #2505 = height of the tallest container
        elif (category == "Special"):
            weight = random.uniform(15, 47.3) #47.3 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 12031) #12032 = length of the container with the most length
            width = random.uniform(2300, 2373) #2374 = width of the widest container
            height = random.uniform(1700, 2652) #2653 = height of the tallest container
        
        origin = random.choice(locations)
        destination = random.choice(locations)
        #Guarantee that origin is different than the destination
        while (origin == destination):
            destination = random.choice(locations)

        random_second_arrival = random.randrange(delta_time_seconds_week) #Random time between the two dates
        new_arrival = start_time + datetime.timedelta(seconds=random_second_arrival)
        
        random_second_deadline = random.randrange(delta_time_seconds_week) #Random time between the two dates
        new_deadline = start_time + datetime.timedelta(seconds=random_second_deadline)
        
        if random_second_deadline > random_second_arrival: #Arrival needs to happen first
            arrival = new_arrival
            deadline = new_deadline
        else:
            arrival = new_deadline
            deadline = new_arrival
        
        freight = Freight(current_id, category, weight, length, width, height, origin, destination, arrival, deadline)
        freight_list.append(freight)
        
        current_id += 1
    
    return freight_list

# Calculate Statistics

In [8]:
def calculateStatistics(trucks):
    total_distance_travelled = 0
    total_distance_travelled_with_no_freight = 0
    total_time_spent = datetime.timedelta(seconds=0)
    total_time_spent_with_no_freight = datetime.timedelta(seconds=0)
    total_grams_of_CO2_emitted = 0

    number_of_freight_delivered = 0

    for truck in trucks:
        timeslots = truck.getTimeSlots()

        for day in days_of_the_week:
            slot = timeslots.get(day)

            if ( (slot) and (not slot.isAvailable()) ):
                total_distance_travelled += slot.getDistanceTravelled()
                total_distance_travelled_with_no_freight += slot.getDistanceTravelledWithNoFreight()
                total_time_spent += slot.getTimeSpent()
                total_time_spent_with_no_freight += slot.getTimeSpentWithNoFreight()
                total_grams_of_CO2_emitted += slot.getGramsOfCO2Emitted()

                number_of_freight_delivered += 1
                
    print("total_distance_travelled:", round (total_distance_travelled, 1), "Km")
    print("total_distance_travelled_with_no_freight:", round(total_distance_travelled_with_no_freight, 1), "Km\n")

    print("total_time_spent:", total_time_spent)
    print("total_time_spent_with_no_freight:", total_time_spent_with_no_freight, end = "\n\n")

    total_grams_of_CO2_emitted = round(total_grams_of_CO2_emitted)
    print("total_grams_of_CO2_emitted:", f"{total_grams_of_CO2_emitted:_}", "grams\n")

    print("number_of_freight_delivered:", number_of_freight_delivered)
    print("percentage of freight delivered:", round( (number_of_freight_delivered / number_of_freight * 100), 1), "%")
    print("\n\n")
    
    distances_travelled.append( round (total_distance_travelled, 1) )
    distances_travelled_with_no_freight.append( round(total_distance_travelled_with_no_freight, 1) )
    grams_of_CO2.append(total_grams_of_CO2_emitted)
    percentage = round( (number_of_freight_delivered / number_of_freight * 100), 1)
    percentage_of_freight_delivered.append(percentage)
    
    distances_travelled_100.append( round (total_distance_travelled * 100 / percentage, 1) )
    distances_travelled_with_no_freight_100.append( round(total_distance_travelled_with_no_freight * 100 / percentage, 1) )
    grams_of_CO2_100.append(total_grams_of_CO2_emitted * 100 / percentage)

In [9]:
def calculate_means_and_std():
    mean_dist = round( np.mean(distances_travelled), 1)
    std_dist = round( np.std(distances_travelled), 1)
    print("mean distance travelled (Km):", mean_dist, "std:", std_dist)
    
    mean_dist_f = round( np.mean(distances_travelled_with_no_freight), 1)
    std_dist_f = round( np.std(distances_travelled_with_no_freight), 1)
    print("mean distance travelled with no freight (Km):", mean_dist_f, "std:", std_dist_f)
    
    mean_CO2 = round( np.mean(grams_of_CO2) )
    std_CO2 = round( np.std(grams_of_CO2) )
    print("mean grams of CO2 emitted:", f"{mean_CO2:_}", "std:", f"{std_CO2:_}")
    
    mean_percentage = round( np.mean(percentage_of_freight_delivered), 1)
    std_percentage = round( np.std(percentage_of_freight_delivered), 1)
    print("\nmean percentage of freight delivered:", mean_percentage, "std:", std_percentage)
    
    print ("\nAssuming that all the freight was delivered instead:")
    
    mean_dist = round( np.mean(distances_travelled_100), 1)
    std_dist = round( np.std(distances_travelled_100), 1)
    print("mean distance travelled (Km):", mean_dist, "std:", std_dist)
    
    mean_dist_f = round( np.mean(distances_travelled_with_no_freight_100), 1)
    std_dist_f = round( np.std(distances_travelled_with_no_freight_100), 1)
    print("mean distance travelled with no freight (Km):", mean_dist_f, "std:", std_dist_f)
    
    mean_CO2 = round( np.mean(grams_of_CO2_100) )
    std_CO2 = round( np.std(grams_of_CO2_100) )
    print("mean grams of CO2 emitted:", f"{mean_CO2:_}", "std:", f"{std_CO2:_}")
    
    print("----------------------------------------------")
    print("\nValues:\n")
    
    print("Distances:")
    for distance in distances_travelled_100:
        print( round(distance, 1) )
        
    print("\nDistances (no freight):")
    for distance in distances_travelled_with_no_freight_100:
        print( round(distance, 1) )
        
    print("\nGrams of CO2:")
    for grams in grams_of_CO2_100:
        print( round(grams) )

# AI Algorithm - Reinforcement Learning

## Create a Custom Environment

In [10]:
class FreightTruckMatchingEnv(gym.Env):
    """Custom Environment that follows gym interface."""

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(2) #Good time slot or bad time slot

        self.observation_space = spaces.Box(low=-1, high=np.inf,
                                            shape=(3,), dtype=np.float64)
        
        self.S2E_reward = Rewards()

    def step(self, action):            
        ###########################Reward the Agent###########################
        
        self.reward = 0.0
        
        if (self.slot): #The slot was available
            self.reward, predicted_concept = self.S2E_reward.return_reward(self.observation, action)
            
            #distance_travelled = self.slot["distance_travelled"]
            #distance_travelled_with_no_freight = self.slot["distance_travelled_with_no_freight"]
            #grams_of_CO2_emitted = self.slot["grams_of_CO2_emitted"]
        
            if (action == 0): #actions: 0 = bad choice, 1 = good choice, according to the agent
                self.bad_slots_truck_list.append( (self.slot, self.trucks[self.truck_ind]) ) #Save it in a list

                #if (distance_travelled > 427):
                    #self.reward += 10.0
                #if (distance_travelled_with_no_freight > 166):
                    #self.reward += 10.0
                #if (grams_of_CO2_emitted > 1_200_000):
                    #self.reward += 10.0

            elif (action == 1): #actions: 0 = bad choice, 1 = good choice, according to the agent
                self.good_slots_truck_list.append( (self.slot, self.trucks[self.truck_ind]) ) #Save it in a list

                #if (distance_travelled < 427):
                    #self.reward += 10.0
                #if(distance_travelled_with_no_freight < 166):
                    #self.reward += 10.0
                #if (grams_of_CO2_emitted < 1_200_000):
                    #self.reward += 10.0
        
            
        ###########################Move to the next truck###########################
        
        self.truck_ind += 1 
        
        if (self.truck_ind >= number_of_trucks): #Every truck covered -> assign freight and move to the next one
            
            ###########################Assign this freight to the timeslot###########################
            
            if (self.good_slots_truck_list): #Check if there is at least one good slot 
                self.slots_truck_list = self.good_slots_truck_list
            elif (self.bad_slots_truck_list): #Else -> Check if there is at least one bad slot 
                self.slots_truck_list = self.bad_slots_truck_list
            
            #Check if there is at least one slot 
            if (self.slots_truck_list): #Slots available -> Let's pick randomly
                best_slot_truck = random.choice(self.slots_truck_list)
                best_truck = best_slot_truck[1]
                best_slot = best_slot_truck[0]

                freight = self.freight_list[self.freight_ind] #Get the current freight
                freight.assignToTruck(best_truck.getTruckId())

                freight_id = self.freight_ind
                freight_destination = freight.getDestination()
                
                timeslot = best_slot["time_slot"]
                distance_travelled = best_slot["distance_travelled"]
                distance_travelled_with_no_freight = best_slot["distance_travelled_with_no_freight"]
                time_spent = best_slot["travel_time"]
                time_spent_with_no_freight = best_slot["time_spent_with_no_freight"]
                grams_of_CO2_emitted = best_slot["grams_of_CO2_emitted"]
                
                timeslot.freightAssignedToTruck(freight_id, distance_travelled, distance_travelled_with_no_freight, time_spent, time_spent_with_no_freight, grams_of_CO2_emitted)

                #Update the truck's location 
                best_truck.updateLocation(freight_destination)
        
            ###########################Move to the next freight###########################
            
            self.slots_truck_list = []
            self.bad_slots_truck_list = []
            self.good_slots_truck_list = []
        
            self.freight_ind += 1
            self.truck_ind = 0
        
            if (self.freight_ind >= number_of_freight): #Every freight covered -> move to the next day
                self.day += 1
                self.freight_ind = 0

                if (self.day >= number_of_days): #We've went through every day
                    self.done = True
                    self.observation = [-1.0, -1.0, -1.0]

                    calculateStatistics(self.trucks)
                
                    self.observation = np.array(self.observation)
                
                    return self.observation, self.reward, self.done, self.truncated, self.info
                
        truck = self.trucks[self.truck_ind] #Get the current truck
        freight = self.freight_list[self.freight_ind] #Get the current freight
        
        if ( not freight.isAssignedToTruck() ):
        
            freight_id = freight.getId()

            freight_category = freight.getCategory()
            freight_weight = freight.getWeight()
            freight_length = freight.getLength()
            freight_width = freight.getWidth()
            freight_height = freight.getHeight()

            freight_arrival = freight.getArrival()
            freight_deadline = freight.getDeadline()
            freight_origin = freight.getOrigin()
            freight_destination = freight.getDestination()
            
            freight_tags = freight.getTags()

            if (truck.containerCanHoldfreight(freight_category, freight_weight, freight_length, freight_width, freight_height, freight_tags)):
                self.slot = truck.timeSlotAvailableForDelivery(self.day, freight_arrival, freight_deadline, freight_origin, freight_destination, freight_weight)
                if (self.slot): #This slot is available
                    
                    #This is the observation
                    self.observation = [ (np.log(self.slot["distance_travelled"] + 1)), (np.log(self.slot["distance_travelled_with_no_freight"] + 1)), (np.log(self.slot["grams_of_CO2_emitted"] + 1)) ]
                    
                else: #This slot is not available
                    self.observation = [-1.0, -1.0, -1.0]
                    self.slot = {}
            else: #The container can't hold the freight
                self.observation = [-1.0, -1.0, -1.0]
                self.slot = {}
        else: #The freight is already assigned
            self.observation = [-1.0, -1.0, -1.0]
            self.slot = {}
                    
        self.observation = np.array(self.observation)
        
        return self.observation, self.reward, self.done, self.truncated, self.info

    def reset(self, seed=None, options=None):
        self.done = False
        self.reward = 0.0
        self.observation = [-1.0, -1.0, -1.0]
        
        self.trucks = generateRandomTrucks(number_of_trucks)
        self.freight_list = generateRandomFreight(number_of_freight)
        
        self.day = 0
        self.freight_ind = 0 #Get the first freight 
        self.truck_ind = 0 #Get The first truck
        
        self.slots_truck_list = []
        self.bad_slots_truck_list = []
        self.good_slots_truck_list = []
                
        ##############################################The first truck and the first freight

        truck = self.trucks[self.truck_ind] #Get the first Truck
        freight = self.freight_list[self.freight_ind] #Get the first freight
        
        freight_id = freight.getId()

        freight_category = freight.getCategory()
        freight_weight = freight.getWeight()
        freight_length = freight.getLength()
        freight_width = freight.getWidth()
        freight_height = freight.getHeight()

        freight_arrival = freight.getArrival()
        freight_deadline = freight.getDeadline()
        freight_origin = freight.getOrigin()
        freight_destination = freight.getDestination()
        
        freight_tags = freight.getTags()
        
        self.slot = {}

        if (truck.containerCanHoldfreight(freight_category, freight_weight, freight_length, freight_width, freight_height, freight_tags)):
            self.slot = truck.timeSlotAvailableForDelivery(self.day, freight_arrival, freight_deadline, freight_origin, freight_destination, freight_weight)
            if (self.slot): #This slot is available
                
                #This is the observation
                self.observation = [ (np.log(self.slot["distance_travelled"] + 1)), (np.log(self.slot["distance_travelled_with_no_freight"] + 1)), (np.log(self.slot["grams_of_CO2_emitted"] + 1)) ]
                
            else: #This slot is not available
                self.observation = [-1.0, -1.0, -1.0]
                self.slot = {}
        else: #The container can't hold the freight
            self.observation = [-1.0, -1.0, -1.0]
            self.slot = {}
                
        
        self.info = {}
        self.truncated = False
        self.observation = np.array(self.observation)
        
        return self.observation, self.info

## Add the environment to the gym registry

In [11]:
gym.register(
    id = "custom/FreightTruckMatching-v1",
    entry_point = FreightTruckMatchingEnv,
    max_episode_steps = number_of_trucks * number_of_freight * number_of_days,
)

## Make sure the environment is valid

In [12]:
#env = FreightTruckMatchingEnv()
env = gym.make("custom/FreightTruckMatching-v1")

# It will check your custom environment and output additional warnings if needed
check_env(env)

## Run it for a couple episodes

In [12]:
#Run it for a couple episodes with random actions to see if there are any errors

episodes = 20

for episode in range(episodes):
    print("\n\n############Episode:", episode, "############\n")
    
    done = False
    obs = env.reset()
    while not done:
        random_action = env.action_space.sample()
        #print("action:", random_action)
        obs, reward, done, truncated, info = env.step(random_action)
        #print('reward:', reward, "\n")



############Episode: 0 ############

total_distance_travelled: 86932.6 Km
total_distance_travelled_with_no_freight: 39192.2 Km

total_time_spent: 36 days, 21:46:00
total_time_spent_with_no_freight: 16 days, 18:17:00

total_grams_of_CO2_emitted: 255_982_516 grams

number_of_freight_delivered: 191
percentage of freight delivered: 76.4 %





############Episode: 1 ############

total_distance_travelled: 93596.2 Km
total_distance_travelled_with_no_freight: 39718.0 Km

total_time_spent: 39 days, 8:40:00
total_time_spent_with_no_freight: 16 days, 17:52:00

total_grams_of_CO2_emitted: 280_272_767 grams

number_of_freight_delivered: 202
percentage of freight delivered: 80.8 %





############Episode: 2 ############

total_distance_travelled: 90489.4 Km
total_distance_travelled_with_no_freight: 41514.5 Km

total_time_spent: 38 days, 3:02:00
total_time_spent_with_no_freight: 17 days, 12:36:00

total_grams_of_CO2_emitted: 247_451_348 grams

number_of_freight_delivered: 190
percentage of freig

## AI Algorithm - PPO

In [17]:
models_dir = f"models/{int(time.time())}/"
logdir = f"logs/{int(time.time())}/"

if not os.path.exists(models_dir):
    os.makedirs(models_dir)

if not os.path.exists(logdir):
    os.makedirs(logdir)

env.reset()

model = PPO('MlpPolicy', env, verbose=0, tensorboard_log=logdir)

TIMESTEPS = 10000
iters = 0
while True:
    iters += 1
    model.learn(total_timesteps=TIMESTEPS, reset_num_timesteps=False, tb_log_name=f"AI ALgorithm 2")
    model.save(f"{models_dir}/{TIMESTEPS*iters}")

total_distance_travelled: 86946.3 Km
total_distance_travelled_with_no_freight: 36845.7 Km

total_time_spent: 36 days, 21:10:00
total_time_spent_with_no_freight: 15 days, 19:20:00

total_grams_of_CO2_emitted: 259_988_394 grams

number_of_freight_delivered: 207
percentage of freight delivered: 82.8 %



total_distance_travelled: 90417.0 Km
total_distance_travelled_with_no_freight: 39511.9 Km

total_time_spent: 37 days, 23:48:00
total_time_spent_with_no_freight: 16 days, 16:37:00

total_grams_of_CO2_emitted: 252_626_899 grams

number_of_freight_delivered: 199
percentage of freight delivered: 79.6 %



total_distance_travelled: 84624.6 Km
total_distance_travelled_with_no_freight: 30326.0 Km

total_time_spent: 35 days, 17:44:00
total_time_spent_with_no_freight: 13 days, 5:25:00

total_grams_of_CO2_emitted: 277_682_007 grams

number_of_freight_delivered: 203
percentage of freight delivered: 81.2 %



total_distance_travelled: 79830.6 Km
total_distance_travelled_with_no_freight: 27885.9 Km

t

KeyboardInterrupt: 

## Load the Model

In [13]:
models_dir = "models/1746023243/"
model_path = f"{models_dir}/1600000.zip"

model = PPO.load(model_path, env=env)

episodes = 30

distances_travelled = []
distances_travelled_with_no_freight = []
grams_of_CO2 = []
percentage_of_freight_delivered = []

distances_travelled_100 = []
distances_travelled_with_no_freight_100 = []
grams_of_CO2_100 = []

for ep in range(episodes):
    obs, _ = env.reset()
    
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)
        
print("\n")
calculate_means_and_std()

total_distance_travelled: 67785.1 Km
total_distance_travelled_with_no_freight: 21166.7 Km

total_time_spent: 29 days, 1:08:00
total_time_spent_with_no_freight: 9 days, 8:15:00

total_grams_of_CO2_emitted: 250_620_434 grams

number_of_freight_delivered: 188
percentage of freight delivered: 75.2 %



total_distance_travelled: 78929.0 Km
total_distance_travelled_with_no_freight: 24164.8 Km

total_time_spent: 33 days, 14:34:00
total_time_spent_with_no_freight: 10 days, 17:32:00

total_grams_of_CO2_emitted: 285_650_276 grams

number_of_freight_delivered: 202
percentage of freight delivered: 80.8 %



total_distance_travelled: 75595.2 Km
total_distance_travelled_with_no_freight: 25946.7 Km

total_time_spent: 32 days, 5:35:00
total_time_spent_with_no_freight: 11 days, 8:26:00

total_grams_of_CO2_emitted: 229_701_587 grams

number_of_freight_delivered: 192
percentage of freight delivered: 76.8 %



total_distance_travelled: 79829.7 Km
total_distance_travelled_with_no_freight: 26134.6 Km

total

total_distance_travelled: 72657.0 Km
total_distance_travelled_with_no_freight: 21180.2 Km

total_time_spent: 30 days, 18:58:00
total_time_spent_with_no_freight: 9 days, 10:04:00

total_grams_of_CO2_emitted: 253_620_511 grams

number_of_freight_delivered: 189
percentage of freight delivered: 75.6 %



total_distance_travelled: 77031.2 Km
total_distance_travelled_with_no_freight: 26184.7 Km

total_time_spent: 32 days, 19:52:00
total_time_spent_with_no_freight: 11 days, 10:43:00

total_grams_of_CO2_emitted: 262_984_627 grams

number_of_freight_delivered: 208
percentage of freight delivered: 83.2 %





mean distance travelled (Km): 75923.7 std: 3962.6
mean distance travelled with no freight (Km): 24581.7 std: 2141.7
mean grams of CO2 emitted: 257_360_730 std: 16_478_049

mean percentage of freight delivered: 80.1 std: 2.8

Assuming that all the freight was delivered instead:
mean distance travelled (Km): 94731.4 std: 3277.3
mean distance travelled with no freight (Km): 30652.9 std: 2150.8

In [ ]:
###IGNORE THE CODE BELOW THIS CELL

In [15]:
models_dir = "models/1746025626/"
model_path = f"{models_dir}/800000.zip"

model = PPO.load(model_path, env=env)

distances_travelled = []
distances_travelled_with_no_freight = []
grams_of_CO2 = []
percentage_of_freight_delivered = []

distances_travelled_100 = []
distances_travelled_with_no_freight_100 = []
grams_of_CO2_100 = []

episodes = 3

for ep in range(episodes):
    obs, _ = env.reset()
    
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)
        
print("\n")
calculate_means_and_std()

total_distance_travelled: 72928.3 Km
total_distance_travelled_with_no_freight: 22618.2 Km

total_time_spent: 31 days, 3:27:00
total_time_spent_with_no_freight: 10 days, 3:43:00

total_grams_of_CO2_emitted: 243_227_949 grams

number_of_freight_delivered: 192
percentage of freight delivered: 76.8 %



total_distance_travelled: 80290.0 Km
total_distance_travelled_with_no_freight: 26108.7 Km

total_time_spent: 34 days, 3:11:00
total_time_spent_with_no_freight: 11 days, 8:11:00

total_grams_of_CO2_emitted: 270_768_756 grams

number_of_freight_delivered: 209
percentage of freight delivered: 83.6 %



total_distance_travelled: 72728.9 Km
total_distance_travelled_with_no_freight: 18426.6 Km

total_time_spent: 31 days, 3:35:00
total_time_spent_with_no_freight: 8 days, 11:37:00

total_grams_of_CO2_emitted: 278_444_312 grams

number_of_freight_delivered: 208
percentage of freight delivered: 83.2 %





mean distance travelled (Km): 75315.7 std: 3518.3
mean distance travelled with no freight (Km):

In [14]:
models_dir = "models/1746027912/"
model_path = f"{models_dir}/1400000.zip"

model = PPO.load(model_path, env=env)

episodes = 3

distances_travelled = []
distances_travelled_with_no_freight = []
grams_of_CO2 = []
percentage_of_freight_delivered = []

distances_travelled_100 = []
distances_travelled_with_no_freight_100 = []
grams_of_CO2_100 = []

for ep in range(episodes):
    obs, _ = env.reset()
    
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)
        
print("\n")
calculate_means_and_std()

total_distance_travelled: 86051.2 Km
total_distance_travelled_with_no_freight: 33241.2 Km

total_time_spent: 36 days, 14:02:00
total_time_spent_with_no_freight: 14 days, 10:42:00

total_grams_of_CO2_emitted: 284_693_258 grams

number_of_freight_delivered: 199
percentage of freight delivered: 79.6 %



total_distance_travelled: 76956.7 Km
total_distance_travelled_with_no_freight: 26919.7 Km

total_time_spent: 33 days, 1:23:00
total_time_spent_with_no_freight: 12 days, 3:46:00

total_grams_of_CO2_emitted: 242_373_261 grams

number_of_freight_delivered: 191
percentage of freight delivered: 76.4 %



total_distance_travelled: 85968.8 Km
total_distance_travelled_with_no_freight: 31274.0 Km

total_time_spent: 36 days, 13:26:00
total_time_spent_with_no_freight: 13 days, 20:51:00

total_grams_of_CO2_emitted: 262_426_932 grams

number_of_freight_delivered: 200
percentage of freight delivered: 80.0 %





mean distance travelled (Km): 82992.2 std: 4267.9
mean distance travelled with no freight (